# Crypto statistical arbitrage | research notebook

I'm not trying to prove crypto is easy money. I want to know whether a stable spread exists and if a simple rule can trade it after costs.

This notebook reads the **saved results** of the Python experiment. Run the command-line pipeline before opening it, and change `run_dir` to the run you want to investigate. The included demo uses fake data.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

run_dir = Path('../results/demo')
with open(run_dir / 'summary.json') as f:
    summary = json.load(f)

print('demo data:', summary['demo'])
print('selected pair:', summary['pair'])
print('metrics:', summary['metrics'])


## Cointegration checks

The testing happens only on the training section. If a pair doesn't pass the stated criteria, I shouldn't force a trade.


In [ ]:
screen = pd.read_csv(run_dir / 'pair_screening_train_only.csv')
cols = ['y', 'x', 'beta', 'pvalue', 'qvalue', 'i1_plausible', 'eligible']
display(screen[cols])


## Out-of-sample equity and risk

The first thing I want to see is the actual sequence of returns and drawdowns. A pretty single number can hide an ugly path.


In [ ]:
trades_path = run_dir / 'out_of_sample_backtest.csv'
if trades_path.exists():
    result = pd.read_csv(trades_path, index_col=0, parse_dates=True)
    ax = result[['equity', 'drawdown']].plot(subplots=True, figsize=(12, 6),
                                           title=['net equity', 'drawdown'], legend=False)
    plt.tight_layout()
    plt.show()
else:
    print('No eligible pair. Read the screen instead of drawing imaginary P&L.')


## What if execution costs are worse?

Cost assumptions need to survive a stress test. The table below does **not** pick the best setting. It just tells me how fragile the strategy is.


In [ ]:
stress_path = run_dir / 'cost_sensitivity.csv'
if stress_path.exists():
    stress = pd.read_csv(stress_path)
    display(stress)
    stress.plot(x='one_way_total_bps', y='net_total_return', marker='o', legend=False)
    plt.axhline(0, color='gray', linestyle='--')
    plt.ylabel('net total return')
    plt.show()


## Research notes

- Did the data support cointegration, or only correlation?
- Do the results depend on a small number of trades?
- How much of the return disappears at higher costs?
- Do these instruments have the borrowing or funding conditions required to trade them?
- What would I check on a new untouched sample before doing anything else?

A failed hypothesis is still useful. I just need to explain it properly.
